In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer

In [32]:
df_world=pd.read_csv("world_data_full_apply(in).csv",index_col=0)
df_world.head(5)

,country,density,abbreviation,agricultural_land,land_area,armed_forces_size,birth_rate,calling_code,capital/major_city,co2-emissions,...,physicians_per_thousand,population,population_labor_force_participation,tax_revenue,total_tax_rate,unemployment_rate,urban_population,latitude,longitude,continent
0,Afghanistan,60.0,AF,58.1,652.230,323.0,32.49,93.0,Kabul,8.672,...,0.28,NaN,48.9,9.3,71.4,11.12,NaN,33.939110,67.709953,Asia
1,Albania,105.0,AL,43.1,28.748,9.0,11.78,355.0,Tirana,4.536,...,1.20,NaN,55.7,18.6,36.6,12.33,NaN,41.153332,20.168331,Europe
2,Algeria,18.0,DZ,17.4,NaN,317.0,24.28,213.0,Algiers,150.006,...,1.72,NaN,41.2,37.2,66.1,11.70,NaN,28.033886,1.659626,Africa
3,Andorra,164.0,AD,40.0,468.000,NaN,7.20,376.0,Andorra la Vella,469.000,...,3.33,77.142,NaN,NaN,NaN,NaN,67.873,42.506285,1.521801,Europe
4,Angola,26.0,AO,47.5,NaN,117.0,40.73,244.0,Luanda,34.693,...,0.21,NaN,77.5,9.2,49.1,6.89,NaN,-11.202692,17.873887,Africa


In [33]:
df_world.info()

<class 'pandas.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 36 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   country                               195 non-null    str    
 1   density                               195 non-null    float64
 2   abbreviation                          188 non-null    str    
 3   agricultural_land                     188 non-null    float64
 4   land_area                             165 non-null    float64
 5   armed_forces_size                     166 non-null    float64
 6   birth_rate                            189 non-null    float64
 7   calling_code                          194 non-null    float64
 8   capital/major_city                    192 non-null    str    
 9   co2-emissions                         183 non-null    float64
 10  cpi                                   175 non-null    float64
 11  cpi_change                    

In [34]:
df_world.isnull().sum() #vemos los nulos esto te dice cuantos valores faltan en cada columna.

country                                   0
density                                   0
abbreviation                              7
agricultural_land                         7
land_area                                30
armed_forces_size                        29
birth_rate                                6
calling_code                              1
capital/major_city                        3
co2-emissions                            12
cpi                                      20
cpi_change                               16
currency-code                            15
fertility_rate                            7
forested_area                             7
gasoline_price                           20
gdp                                       2
gross_primary_education_enrollment        7
gross_tertiary_education_enrollment      12
infant_mortality                          6
largest_city                              6
life_expectancy                           8
maternal_mortality_ratio        

In [35]:
(df_world.isnull().mean()*100).sort_values(ascending=False) #aqui ordenamos en porcenjate y ascendente vemos que los dos primeros tienen un indice de nulos muy alto.vamos a ver que tipo de dqatos es ambos.

population                              80.000000
urban_population                        76.923077
minimum_wage                            23.076923
land_area                               15.384615
armed_forces_size                       14.871795
tax_revenue                             13.333333
cpi                                     10.256410
gasoline_price                          10.256410
unemployment_rate                        9.743590
population_labor_force_participation     9.743590
cpi_change                               8.205128
currency-code                            7.692308
maternal_mortality_ratio                 7.179487
total_tax_rate                           6.153846
co2-emissions                            6.153846
gross_tertiary_education_enrollment      6.153846
life_expectancy                          4.102564
physicians_per_thousand                  3.589744
abbreviation                             3.589744
agricultural_land                        3.589744


In [11]:
df_world[["population","urban_population"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   population        39 non-null     float64
 1   urban_population  45 non-null     float64
dtypes: float64(2)
memory usage: 3.2 KB


In [ ]:
df_world[["population","urban_population"]].describe() # en estas dos columnas el porcentaje de nulos es tan grande que no podriamos sacar una conclusion real, al ser numerico podriamos imputar pero tampoco seria muy fiable.

,population,urban_population
count,39.000000,45.000000
mean,333.542718,279.461044
std,296.467504,283.322352
min,10.084000,5.464000
25%,87.130000,40.765000
50%,215.056000,179.039000
75%,556.162500,417.765000
max,973.560000,984.812000


In [ ]:
# creo que la mejor solucion sera eliminarlo aunque me voy a guardar en una variable esta informacion por si la necesito en algun momento para comprobar alguna cosa puntual.

In [37]:
df_world['population_missing'] = df_world['population'].isnull()

In [ ]:
#vamos a analizar en produndidad esos patrones.
df_world.groupby('continent')['population_missing'].mean().sort_values(ascending=False) # no hay un patron en concreto que lo relacione.

continent
North America      1.000000
Asia               0.933333
Africa             0.924528
South America      0.846154
Europe             0.808511
Central America    0.550000
Oceania            0.214286
Name: population_missing, dtype: float64

In [13]:
columnas_a_eliminar=["population","urban_population"] #las meto en una variable por si acaso necesito algo de estas columnas en algun momento.

In [ ]:
df_world_eliminadas=df_world[columnas_a_eliminar].copy() # creamos un dataframe totalmente independiente que no interfiere con los otros. asi no me va a modificar el dataframe real.

In [29]:
df_world.head().T

,0,1,2,3,4
country,Afghanistan,Albania,Algeria,Andorra,Angola
density,60.0,105.0,18.0,164.0,26.0
abbreviation,AF,AL,DZ,AD,AO
agricultural_land,58.1,43.1,17.4,40.0,47.5
land_area,652.23,28.748,NaN,468.0,NaN
armed_forces_size,323.0,9.0,317.0,NaN,117.0
birth_rate,32.49,11.78,24.28,7.2,40.73
calling_code,93.0,355.0,213.0,376.0,244.0
capital/major_city,Kabul,Tirana,Algiers,Andorra la Vella,Luanda
co2-emissions,8.672,4.536,150.006,469.0,34.693


In [ ]:
#vamos con las siguientes columnas :minimum_wage                            23.076923
#                                   land_area  